# 01 · Data exploration

Explores the **processed** dataset produced by `scripts/prepare_data.py`.
Set `DATASET` below (`nbaiot`, `ciciot2023`, `ton_iot`, `bot_iot` or `synthetic`).

> The *synthetic* dataset is a pipeline-testing artefact, not real traffic.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import ppfl  # noqa: F401  (Windows OpenMP import-order guard)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ppfl.visualization.plots import set_style
set_style()
%matplotlib inline

In [ ]:
DATASET = "nbaiot"   # "synthetic" to explore the test data
from ppfl.data.loader import load_processed, META_COLS
frame, meta = load_processed(ROOT / "data" / "processed" / DATASET)
features = [c for c in frame.columns if c not in META_COLS]
print(f"{len(frame):,} rows x {len(features)} features")
meta.get("cleaning")

## Class and device composition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
frame["label"].value_counts().plot.barh(ax=axes[0], color="#4c72b0")
axes[0].set(title="Rows per class", xlabel="Rows")
pd.crosstab(frame["device"], frame["label"] == "benign").rename(columns={True: "benign", False: "attack"}).plot.barh(stacked=True, ax=axes[1])
axes[1].set(title="Benign vs attack rows per device", xlabel="Rows")
plt.tight_layout()

## Feature statistics
Heavy tails motivate standardisation (and optionally `data.log_transform`).

In [ ]:
desc = frame[features].describe().T
desc["skew"] = frame[features].skew()
desc.sort_values("skew", ascending=False).head(15)

In [ ]:
sample = frame.sample(min(len(frame), 20000), random_state=0)
plt.figure(figsize=(9, 7.5))
sns.heatmap(sample[features].corr(), cmap="coolwarm", center=0, xticklabels=False, yticklabels=False)
plt.title("Feature correlation (sample)")

## Benign vs attack traffic
A 2-D PCA projection fitted on benign traffic shows how attack families depart from normal behaviour.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
X = sample[features].fillna(sample[features].mean()).to_numpy()
benign = sample["label"].to_numpy() == "benign"
scaler = StandardScaler().fit(X[benign])
pca = PCA(2).fit(scaler.transform(X[benign]))
Z = pca.transform(scaler.transform(X))
plt.figure(figsize=(8, 6))
for label in sorted(sample["label"].unique()):
    m = sample["label"].to_numpy() == label
    plt.scatter(Z[m, 0], Z[m, 1], s=4, alpha=0.4, label=label)
plt.legend(markerscale=4, fontsize=8, bbox_to_anchor=(1.01, 1))
plt.title("PCA (fitted on benign rows)"); plt.xlabel("PC1"); plt.ylabel("PC2")